# Sigmoid Backpropagation Examples

NumPy/TensorFlow exercises in sigmoid networks and gradient updates.

Stage: learning experiment.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the sigmoid activation function and its derivative
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# Set random seed for reproducibility
np.random.seed(42)

# Define the neural network architecture
input_size = 2
hidden1_size = 2
hidden2_size = 3
output_size = 1

# Initialize weights and biases
weights_input_hidden1 = np.random.rand(input_size, hidden1_size)
biases_hidden1 = np.zeros((1, hidden1_size))
weights_hidden1_hidden2 = np.random.rand(hidden1_size, hidden2_size)
biases_hidden2 = np.zeros((1, hidden2_size))
weights_hidden2_output = np.random.rand(hidden2_size, output_size)
biases_output = np.zeros((1, output_size))

# Hyperparameters
learning_rate = 0.1
epochs = 100000
batch_size = 1  # Mini-batch size

# Training data
X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_train = np.array([[1], [0], [0], [1]])

# Training loop
mse_history = []

for epoch in range(epochs):
    mse_epoch = 0

    # Shuffle the training data for each epoch
    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]

    for i in range(len(X_train_shuffled)):
        # Forward pass
        hidden1_input = np.dot(X_train_shuffled[i], weights_input_hidden1) + biases_hidden1
        hidden1_output = sigmoid(hidden1_input)
        hidden2_input = np.dot(hidden1_output, weights_hidden1_hidden2) + biases_hidden2
        hidden2_output = sigmoid(hidden2_input)
        output = sigmoid(np.dot(hidden2_output, weights_hidden2_output) + biases_output)

        # Calculate Mean Squared Error (MSE)
        mse = np.mean((output - y_train_shuffled[i])**2)
        mse_epoch += mse

        # Backward pass
        output_error = output - y_train_shuffled[i]
        output_delta = output_error * sigmoid_derivative(output)
        hidden2_error = np.dot(output_delta, weights_hidden2_output.T)
        hidden2_delta = hidden2_error * sigmoid_derivative(hidden2_output)
        hidden1_error = np.dot(hidden2_delta, weights_hidden1_hidden2.T)
        hidden1_delta = hidden1_error * sigmoid_derivative(hidden1_output)

        # Update weights and biases
        weights_hidden2_output -= learning_rate * np.dot(hidden2_output.T, output_delta)
        biases_output -= learning_rate * output_delta
        weights_hidden1_hidden2 -= learning_rate * np.dot(hidden1_output.T, hidden2_delta)
        biases_hidden2 -= learning_rate * hidden2_delta
        weights_input_hidden1 -= learning_rate * np.dot(X_train_shuffled[i].reshape(-1, 1), hidden1_delta)
        biases_hidden1 -= learning_rate * hidden1_delta

    # Calculate mean MSE for the epoch
    mse_epoch /= len(X_train_shuffled)
    mse_history.append(mse_epoch)

# Plot the Mean Squared Error (MSE) against the number of epochs
plt.plot(range(1, epochs + 1), mse_history)
plt.title('Mean Squared Error (MSE) vs. Number of Epochs (SGD)')
plt.xlabel('Number of Epochs')
plt.ylabel('Mean Squared Error (MSE)')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf  # Import TensorFlow

# Define the sigmoid activation function and its derivative
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# Set random seed for reproducibility
np.random.seed(42)

# Define the neural network architecture
input_size = 2
hidden1_size = 2
hidden2_size = 3
output_size = 1

# Initialize weights and biases
weights_input_hidden1 = np.random.rand(input_size, hidden1_size)
biases_hidden1 = np.zeros((1, hidden1_size))
weights_hidden1_hidden2 = np.random.rand(hidden1_size, hidden2_size)
biases_hidden2 = np.zeros((1, hidden2_size))
weights_hidden2_output = np.random.rand(hidden2_size, output_size)
biases_output = np.zeros((1, output_size))

# Hyperparameters
learning_rate = 0.1
epochs = 100000
batch_size = 4

# Create Adam optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)  # Use Adam optimizer

# Training data
X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_train = np.array([[1], [0], [0], [1]])

# Training loop
mse_history = []

for epoch in range(epochs):
    mse_epoch = 0

    # Shuffle the training data for each epoch
    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]

    # Mini-batch training
    for i in range(0, len(X_train_shuffled), batch_size):
        mini_batch_X = X_train_shuffled[i:i+batch_size]
        mini_batch_y = y_train_shuffled[i:i+batch_size]

        # Forward pass
        hidden1_input = np.dot(mini_batch_X, weights_input_hidden1) + biases_hidden1
        hidden1_output = sigmoid(hidden1_input)
        hidden2_input = np.dot(hidden1_output, weights_hidden1_hidden2) + biases_hidden2
        hidden2_output = sigmoid(hidden2_input)
        output = sigmoid(np.dot(hidden2_output, weights_hidden2_output) + biases_output)

        # Calculate Mean Squared Error (MSE)
        mse = np.mean((output - mini_batch_y)**2)
        mse_epoch += mse

        # Backward pass
        output_error = output - mini_batch_y
        output_delta = output_error * sigmoid_derivative(output)
        hidden2_error = np.dot(output_delta, weights_hidden2_output.T)
        hidden2_delta = hidden2_error * sigmoid_derivative(hidden2_output)
        hidden1_error = np.dot(hidden2_delta, weights_hidden1_hidden2.T)
        hidden1_delta = hidden1_error * sigmoid_derivative(hidden1_output)

        # Update weights and biases using Adam optimizer
        with tf.GradientTape() as tape:
            weights_input_hidden1_var = tf.Variable(weights_input_hidden1)
            biases_hidden1_var = tf.Variable(biases_hidden1)
            weights_hidden1_hidden2_var = tf.Variable(weights_hidden1_hidden2)
            biases_hidden2_var = tf.Variable(biases_hidden2)
            weights_hidden2_output_var = tf.Variable(weights_hidden2_output)
            biases_output_var = tf.Variable(biases_output)

            hidden1_input_var = tf.matmul(mini_batch_X, weights_input_hidden1_var) + biases_hidden1_var
            hidden1_output_var = sigmoid(hidden1_input_var)
            hidden2_input_var = tf.matmul(hidden1_output_var, weights_hidden1_hidden2_var) + biases_hidden2_var
            hidden2_output_var = sigmoid(hidden2_input_var)
            output_var = sigmoid(tf.matmul(hidden2_output_var, weights_hidden2_output_var) + biases_output_var)

            loss = tf.reduce_mean((output_var - mini_batch_y) ** 2)

        gradients = tape.gradient(loss, [weights_input_hidden1_var, biases_hidden1_var, weights_hidden1_hidden2_var, biases_hidden2_var, weights_hidden2_output_var, biases_output_var])
        optimizer.apply_gradients(zip(gradients, [weights_input_hidden1_var, biases_hidden1_var, weights_hidden1_hidden2_var, biases_hidden2_var, weights_hidden2_output_var, biases_output_var]))

        weights_input_hidden1 = weights_input_hidden1_var.numpy()
        biases_hidden1 = biases_hidden1_var.numpy()
        weights_hidden1_hidden2 = weights_hidden1_hidden2_var.numpy()
        biases_hidden2 = biases_hidden2_var.numpy()
        weights_hidden2_output = weights_hidden2_output_var.numpy()
        biases_output = biases_output_var.numpy()

    # Calculate mean MSE for the epoch
    mse_epoch /= len(X_train_shuffled) / batch_size
    mse_history.append(mse_epoch)

# Plot the Mean Squared Error (MSE) against the number of epochs
plt.plot(range(1, epochs + 1), mse_history)
plt.title('Mean Squared Error (MSE) vs. Number of Epochs (GD with Adam optimizer)')
plt.xlabel('Number of Epochs')
plt.ylabel('Mean Squared Error (MSE)')
plt.show()
